# Temporary notebook for abTEM simulation

CHL will be handling this and transfer it to `simu_abtem.py` when it's done

In [ ]:
def simulate(x,tX,tY):
    #----variables----
    #x: thickness in angstrom
    #start = datetime.now()
    
    numUC = int(x/dx)#number of unit cells in z direction (variable x)
    #tag = '/vac_conv19p1_volt200kV_tilt0each_{:04d}.tiff'.format(int(numUC*dx))
    sc = atoms * (16,16,numUC) #todo: may need to do the rounding instead
    #sc.center(axis=(0,1),vacuum=10)
    
    #fp = FrozenPhonons(sc,20,{'Sr':.1,'Ti':.1,'O':.1},seed=1)
    fp = FrozenPhonons(sc,20,{'Sr':.088,'Ti':.0746,'O':.0963},seed=1)
    potential = Potential(fp,gpts=512, projection='infinite',#
                          #sampling=0.15, projection='infinite', #,sampling=0.06
                          slice_thickness= 2,storage  = 'gpu', precalculate=True,
                          device='gpu', parametrization='kirkland')
    probe = Probe(energy=200e3, semiangle_cutoff=19.1,tilt=(tX,tY),
                  device='gpu')
    probe.grid.match(potential)
    # print(probe.cutoff_scattering_angles)
    
    pixelated_detector = PixelatedDetector(max_angle=45) #temporary limit for thickness
 #29.345 33.258 
    gridscan = GridScan(start=[29.345,29.345], end=[33.258 ,33.258],sampling=.3)
    #scan = GridScan(start=[19.7,19.7], end=[27.6,27.6],sampling=.2)

    pixelated_measurement = probe.scan(gridscan, pixelated_detector, potential,pbar=False)
    pacbed = np.mean(pixelated_measurement.array,axis=(0,1)).astype(np.float32)
    
    #end = datetime.now() 
    # print('Time elapsed (hh:mm:ss.ms):  {}  || thickness: {} $\AA$  || shape: {}'.format(end-start,
    #                                                                                     numUC*dx,
                                                                                        # pacbed.shape))
    
    return pacbed